# MS lesion segmentation — benchmark

**Author:** micmas (Neurodesk MS Workshop)

**Date:** June 2026

**License:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

---

Companion to `ms_workshop.ipynb`. This notebook **scans the results folder, finds every subject that already has SAMSEG and/or LST-AI output**, scores each against the expert consensus mask (Dice), and summarises the comparison.

It runs **no processing** — it only reads results produced by the main notebook, so it is fast. Point `WORK` at the folder that holds your `derivatives/` (e.g. the shared teaching folder).

## Table of contents
[1. Setup](#1.-Setup)  
[2. Discover and score](#2.-Discover-all-processed-subjects-and-score-them)  
[3. Summary](#3.-Summary-and-comparison)  
[4. Clinical scores](#4.-Add-clinical-scores-(optional))  
[5. Save / share](#5.-Save-and-share-the-results)

## 1. Setup

In [ ]:
from pathlib import Path
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn.image import resample_to_img
from IPython.display import display
%matplotlib inline

import warnings
warnings.filterwarnings('ignore', message='.*no sform.*')

# ============================ Configuration ============================
WORK = Path('/home/jovyan/neurodesktop-storage/neurodesk-ms-workshop')
# WORK = Path('/data/teaching/micmas/neurodesk-ms-workshop')   # pre-processed teaching folder

DATA = WORK / 'open_ms_data'     # holds the expert masks
OUT  = WORK / 'derivatives'      # holds SAMSEG / LST-AI results
print('WORK:', WORK); print('DATA:', DATA); print('OUT :', OUT)

# --- helpers ---------------------------------------------------------------
def grab(folder, *keys):
    for f in sorted(Path(folder).glob('*.nii.gz')):
        n = f.name.lower()
        if all(k in n for k in keys):
            return f
    return None

def find_gt_for(name):
    # locate the expert consensus mask for a subject across the dataset roots
    cs = DATA / 'cross_sectional'
    for root in [cs/'coregistered_resampled', cs/'coregistered', cs]:
        d = root / name
        if d.is_dir():
            gt = (grab(d,'consensus') or grab(d,'gold') or grab(d,'lesion')
                  or grab(d,'gt') or grab(d,'mask'))
            if gt:
                return gt
    return None

def dice(a, b):
    inter = np.logical_and(a, b).sum(); denom = a.sum() + b.sum()
    return 2.0 * inter / denom if denom else float('nan')

def metrics(mask_img, gt_img, gt_d):
    native = mask_img.get_fdata() > 0
    vox = float(np.prod(mask_img.header.get_zooms()[:3]))
    vol_ml = native.sum() * vox / 1000.0
    in_gt = resample_to_img(mask_img, gt_img, interpolation='nearest').get_fdata() > 0
    return round(dice(in_gt, gt_d), 3), round(vol_ml, 1)

print('Setup complete.')

## 2. Discover all processed subjects and score them

In [ ]:
# Build the SAMSEG / LST-AI lesion masks from whatever each subject folder contains.
def samseg_mask(subj_out):
    les = subj_out / 'samseg' / 'lesions.nii.gz'
    if les.exists():
        return nib.load(str(les))
    seg = subj_out / 'samseg' / 'seg.mgz'           # derive lesion label (99) directly, no FreeSurfer needed
    if seg.exists():
        s = nib.load(str(seg))
        return nib.Nifti1Image((s.get_fdata() == 99).astype('uint8'), s.affine, s.header)
    return None

def lstai_mask(subj_out):
    d = subj_out / 'lst_ai'
    if not d.is_dir():
        return None
    p = grab(d, 'seg') or grab(d, 'lesion') or next(iter(sorted(d.glob('*.nii.gz'))), None)
    return nib.load(str(p)) if p else None

records = []
for subj_out in sorted(p for p in OUT.iterdir() if p.is_dir()):
    name = subj_out.name
    gt = find_gt_for(name)
    if gt is None:
        continue                                    # no expert mask -> can't score
    gt_img = nib.load(str(gt)); gt_d = gt_img.get_fdata() > 0
    vox = float(np.prod(gt_img.header.get_zooms()[:3]))
    row = {'id': name, 'expert_ml': round(gt_d.sum() * vox / 1000.0, 1)}

    sam = samseg_mask(subj_out)
    if sam is not None:
        row['SAMSEG_dice'], row['SAMSEG_ml'] = metrics(sam, gt_img, gt_d)
    lst = lstai_mask(subj_out)
    if lst is not None:
        row['LST-AI_dice'], row['LST-AI_ml'] = metrics(lst, gt_img, gt_d)

    if 'SAMSEG_dice' in row or 'LST-AI_dice' in row:
        records.append(row)

bench = pd.DataFrame(records).sort_values('id').reset_index(drop=True)
print(f'Found {len(bench)} subject(s) with results in {OUT}')
bench

## 3. Summary and comparison

In [ ]:
methods = [m for m in ['SAMSEG', 'LST-AI'] if f'{m}_dice' in bench.columns]

summary = pd.DataFrame(
    {m: [bench[f'{m}_dice'].mean(), bench[f'{m}_dice'].std(), bench[f'{m}_dice'].count()] for m in methods},
    index=['mean Dice', 'sd', 'n']
).round(3)
print(summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# boxplot of Dice per method
axes[0].boxplot([bench[f'{m}_dice'].dropna() for m in methods])
axes[0].set_xticks(range(1, len(methods) + 1)); axes[0].set_xticklabels(methods)
axes[0].set_ylabel('Dice vs expert'); axes[0].set_title(f'Dice across {len(bench)} subjects')

# per-subject grouped bars
x = np.arange(len(bench)); w = 0.8 / max(len(methods), 1)
for k, m in enumerate(methods):
    axes[1].bar(x + (k - (len(methods) - 1) / 2) * w, bench[f'{m}_dice'], width=w, label=m)
axes[1].set_xticks(x); axes[1].set_xticklabels(bench['id'], rotation=90, fontsize=7)
axes[1].set_ylabel('Dice vs expert'); axes[1].legend(); axes[1].set_title('Per-subject Dice')

plt.tight_layout(); plt.show()

## 4. Add clinical scores (optional)

In [ ]:
# Join the clinical data shipped with the dataset (age, sex, MS type, EDSS).
out_csv = OUT / 'benchmark.csv'
demog_path = DATA / 'cs_demog.rda'
if demog_path.exists():
    import pyreadr
    demog = pyreadr.read_r(str(demog_path))['cs_demog']
    table = bench.merge(demog, on='id', how='left')
else:
    print('cs_demog.rda not found — saving the Dice table without clinical data.')
    table = bench

display(table)
table.to_csv(out_csv, index=False)
print('Saved', out_csv)

## 5. Save and share the results

The benchmark table is written to `derivatives/benchmark.csv`. To make the processed results reusable (so the notebooks skip the slow steps next time), copy them into the shared teaching folder.

Run these in a **terminal** (adjust the paths if yours differ):

```bash
# Source = where you generated the results; DEST = shared teaching folder
SRC=/home/jovyan/neurodesktop-storage/neurodesk-ms-workshop
DEST=/data/teaching/micmas/neurodesk-ms-workshop
mkdir -p "$DEST"

# Copy the generated results (and the dataset) into the teaching folder.
# -n = never overwrite files that are already there.
cp -rn "$SRC/derivatives"   "$DEST/"
cp -rn "$SRC/open_ms_data"  "$DEST/"
```

To **move** instead of copy (frees space on your home storage), replace the two `cp -rn` lines with:

```bash
mv "$SRC/derivatives"  "$DEST/"
mv "$SRC/open_ms_data" "$DEST/"
```

Afterwards, set `WORK = Path('/data/teaching/micmas/neurodesk-ms-workshop')` at the top of either notebook and everything is read from the pre-computed results.